# AVX integrator

It appears that the integration routine requires the most time in the MPC problem.
This is because many small computations need to be performed sequentially.
However, many of these sequential computations can be performed in parallel, or in parallel with some time delay.
JAX probably has difficulty finding (and exploiting) this structure in a general program, so extra care probably needs to be taken with the programmer.
The following attempts this, in part by forming a giant (block) iteration matrix.

FYI: by AVX, I'm referencing Intel's [AVX](https://en.wikipedia.org/wiki/Advanced_Vector_Extensions).
However, I really mean architecture specific SIMD instructions, e.g., [ARM NEON][https://www.arm.com/technologies/neon].
I'll see if this improves performance...

In [ ]:
%matplotlib ipympl

import jax
jax.config.update("jax_enable_x64", True)


In [ ]:
import functools

import jax.numpy as jnp
import numpy as np
import scipy.linalg as sci_lin

import exp_mpc.stewart_min.const as const
import exp_mpc.stewart_min.utils as utils
import exp_mpc.stewart_min.comp as comp

In [ ]:
n = 200
rstate_num = 12
vstate_num = 3 * const.E0_acc.shape[0] + 3 * const.E0_omega.shape[0]
control_num = 6

acc_ref = jnp.array([0.5, -0.2, 0.1]) + const.gravity
acc_ref = jnp.tile(A=acc_ref, reps=(n, 1))
omega_ref = jnp.array([0.2, 0.1, -0.3])
omega_ref = jnp.tile(A=omega_ref, reps=(n, 1))

In [ ]:
np.random.seed(42)
rstate0 = jnp.zeros(rstate_num)
vstate0_irl = jnp.array(np.random.uniform(-0.5, 0.5, vstate_num))
vstate0_sim = jnp.array(np.random.uniform(-0.5, 0.5, vstate_num))
control0 = jnp.array(np.random.uniform(-0.5, 0.5, control_num))
control = utils.Control(jnp.array(np.random.uniform(-1.0, 1.0, control_num * n)).reshape(-1, control_num))

## initial benchmarking

This shouldn't involve many SIMD computations.

In [ ]:
get_rstate = utils.get_rstate
get_vstate_irl = utils.get_vstate_irl
get_vstate = utils.get_vstate

# get_rstate = jax.jit(utils.get_rstate)
# get_vstate_irl = jax.jit(utils.get_vstate_irl)
# get_vstate = jax.jit(utils.get_vstate)

@jax.jit
def zero_vstate() -> utils.VState:
    return utils.VState(jnp.zeros(vstate_num), jnp.zeros(6))

@jax.jit
def tracking_error(
    acc_ref: jax.Array,
    omega_ref: jax.Array,
    rstate0: jax.Array,
    vstate0_irl: jax.Array,
    vstate0_sim: jax.Array,
    control0: jax.Array,
    control: utils.Control,
) -> jax.Array:
    rstate = get_rstate(control, rstate0)

    vstate_irl = get_vstate_irl(rstate, control, control0, vstate0_irl)
    vstate_sim = get_vstate(acc_ref, omega_ref, vstate0_sim)

    diff = jnp.ravel(vstate_irl.y_state[1:] - vstate_sim.y_state[1:])
    # return jnp.mean(jnp.square(diff))
    return diff @ diff

@jax.jit
def tracking_error_flat(
    acc_ref: jax.Array,
    omega_ref: jax.Array,
    rstate0: jax.Array,
    vstate0_irl: jax.Array,
    vstate0_sim: jax.Array,
    control0: jax.Array,
    control_flat: jax.Array,
) -> jax.Array:
    return tracking_error(
        acc_ref=acc_ref,
        omega_ref=omega_ref,
        rstate0=rstate0,
        vstate0_irl=vstate0_irl,
        vstate0_sim=vstate0_sim,
        control0=control0,
        control=utils.Control.from_flat(control_flat),
    )

tracking_error_and_grad = jax.jit(jax.value_and_grad(
    tracking_error_flat, argnums=6
))

def call_ref() -> jax.Array:
    res = tracking_error(
        acc_ref=acc_ref,
        omega_ref=omega_ref,
        rstate0=rstate0,
        vstate0_irl=vstate0_irl,
        vstate0_sim=vstate0_sim,
        control0=control0,
        control=control,
    )
    res.block_until_ready()
    return res

def call_ref_grad() -> jax.Array:
    res = tracking_error_and_grad(
        acc_ref,
        omega_ref,
        rstate0,
        vstate0_irl,
        vstate0_sim,
        control0,
        control.flatten(),
    )
    res[0].block_until_ready()
    res[1].block_until_ready()
    return res

In [ ]:
# call_ref()
# with jax.profiler.trace("/tmp/jax-trace", create_perfetto_link=True):
#     call_ref()

In [ ]:
call_ref()  # compile
%timeit -n 100 -r 7 call_ref()

In [ ]:
call_ref_grad()  # compile
%timeit -n 100 -r 7 call_ref_grad()

## monolithic numpy loop

Implement `get_rstate`, `get_vstate_irl`, and `get_vstate` in one for monolithic for loop, hopefully for better SIMD performance?
First, we implement a numpy version for easier debugging.

In [ ]:
# full vesetibular models
E0_ves = sci_lin.block_diag(*([const.E0_acc] * 3 + [const.E0_omega] * 3))
E1_ves = sci_lin.block_diag(*([const.E1_acc] * 3 + [const.E1_omega] * 3))
C_ves = sci_lin.block_diag(*([const.C_acc] * 3 + [const.C_omega] * 3))

# integration matrices for double integrator dynamics
E0_dyn = np.array(
    [
        [1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.005, 0.0, 0.0, 0.0, 0.0, 0.0],
        [0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.005, 0.0, 0.0, 0.0, 0.0],
        [0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.005, 0.0, 0.0, 0.0],
        [0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.005, 0.0, 0.0],
        [0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.005, 0.0],
        [0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.005],
        [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0],
        [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0],
        [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0],
        [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0],
        [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0],
        [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0],
    ]
)
E1_dyn = np.array(
    [
        [1.25e-05, 0.00e00, 0.00e00, 0.00e00, 0.00e00, 0.00e00],
        [0.00e00, 1.25e-05, 0.00e00, 0.00e00, 0.00e00, 0.00e00],
        [0.00e00, 0.00e00, 1.25e-05, 0.00e00, 0.00e00, 0.00e00],
        [0.00e00, 0.00e00, 0.00e00, 1.25e-05, 0.00e00, 0.00e00],
        [0.00e00, 0.00e00, 0.00e00, 0.00e00, 1.25e-05, 0.00e00],
        [0.00e00, 0.00e00, 0.00e00, 0.00e00, 0.00e00, 1.25e-05],
        [5.00e-03, 0.00e00, 0.00e00, 0.00e00, 0.00e00, 0.00e00],
        [0.00e00, 5.00e-03, 0.00e00, 0.00e00, 0.00e00, 0.00e00],
        [0.00e00, 0.00e00, 5.00e-03, 0.00e00, 0.00e00, 0.00e00],
        [0.00e00, 0.00e00, 0.00e00, 5.00e-03, 0.00e00, 0.00e00],
        [0.00e00, 0.00e00, 0.00e00, 0.00e00, 5.00e-03, 0.00e00],
        [0.00e00, 0.00e00, 0.00e00, 0.00e00, 0.00e00, 5.00e-03],
    ]
)

E0_dyn_euler = np.eye(12)
for i in range(6):
    E0_dyn_euler[0 + i, 6 + i] = const.dt
E1_dyn_euler = np.zeros((12, 6))
for i in range(6):
    E1_dyn_euler[0 + i, 0 + i] = const.dt**2
    E1_dyn_euler[6 + i, 0 + i] = const.dt

# integration matrices for full dynamics
# E0 = sci_lin.block_diag(E0_dyn, E0_ves, E0_ves)
# E1 = sci_lin.block_diag(E1_dyn, E1_ves, E1_ves)
E0 = sci_lin.block_diag(E0_dyn_euler, E0_ves, E0_ves)
E1 = sci_lin.block_diag(E1_dyn_euler, E1_ves, E1_ves)

In [ ]:
def monolithic_tracking_error(
    acc_ref: np.ndarray,
    omega_ref: np.ndarray,
    rstate0: np.ndarray,
    vstate0_irl: np.ndarray,
    vstate0_sim: np.ndarray,
    control0: np.ndarray,
    control: np.ndarray,
) -> float:
    # the more subtle assertions...
    assert acc_ref.shape == omega_ref.shape
    assert acc_ref.shape[0] == control.shape[0]
    assert control0.shape == (control.shape[1],)
    assert rstate0.shape == (12,)
    assert vstate0_irl.shape == (15,)
    assert vstate0_sim.shape == (15,)

    n = acc_ref.shape[0]  # horizon num

    x0 = np.concatenate([rstate0, vstate0_irl, vstate0_sim])
    x_res = np.empty((n + 1, x0.size))
    x_res[0] = x0

    # append initial control, which is only applied to vstate_irl
    control = np.vstack([control0.reshape(1, -1), control])

    def comp_uv_irl(x: np.ndarray, u: np.ndarray) -> np.ndarray:
        assert x.shape == (42,)
        assert u.shape == (6,)

        xr = x[:12]

        euler = xr[3: 6]
        euler_dot = xr[9: 12]
        R = comp.rot(*euler)

        u_acc = np.ravel(R.T @ (u[:3] + const.gravity))
        u_omega = np.ravel(comp.angle_vel(*np.concatenate([euler, euler_dot])))

        return np.concatenate([u_acc, u_omega])

    for i in range(1, x_res.shape[0]):
        ur = control[i]
        uv_irl = comp_uv_irl(x_res[i - 1], control[i - 1])
        uv_sim = np.concatenate([acc_ref[i - 1], omega_ref[i - 1]])
        u = np.concatenate([ur, uv_irl, uv_sim])
        assert u.shape == (18,)

        x_res[i] = E0 @ x_res[i - 1] + E1 @ u

    xv_sim = x_res[:, 12 + 15:]
    xv_irl = np.empty((n + 1, 15))
    xv_irl[:-1, :] = x_res[1:, 12: 12 + 15]
    uv_irl = comp_uv_irl(x_res[-1], control[-1])
    xv_irl[-1] = E0_ves @ xv_irl[-2] + E1_ves @ uv_irl

    yv_irl = np.transpose(C_ves @ xv_irl.T)
    yv_sim = np.transpose(C_ves @ xv_sim.T)

    # xr = x_res[:, :12]
    # rstate = utils.RState(jnp.array(xr))
    # vstate_irl = utils.VState(jnp.array(xv_irl), jnp.array(yv_irl))
    # vstate_sim = utils.VState(jnp.array(xv_sim), jnp.array(yv_sim))
    diff = np.ravel(yv_irl[1:] - yv_sim[1:])
    return float(diff @ diff)

def call_np() -> float:
    res = monolithic_tracking_error(
        acc_ref=np.array(acc_ref),
        omega_ref=np.array(omega_ref),
        rstate0=np.array(rstate0),
        vstate0_irl=np.array(vstate0_irl),
        vstate0_sim=np.array(vstate0_sim),
        control0=np.array(control0),
        control=np.array(control.control),
    )
    return res

In [ ]:
check0 = call_ref()
check1 = call_np()
check0 - check1

In [ ]:
%timeit -n 100 -r 7 call_np()

## monolithic jax loop

In [ ]:
@jax.jit
def monolithic_tracking_error_jax(
    acc_ref: jax.Array,
    omega_ref: jax.Array,
    rstate0: jax.Array,
    vstate0_irl: jax.Array,
    vstate0_sim: jax.Array,
    control0: jax.Array,
    control: utils.Control,
) -> jax.Array:
    n = acc_ref.shape[0]  # horizon num

    x0 = jnp.concatenate([rstate0, vstate0_irl, vstate0_sim])
    x_res = jnp.empty((n + 1, x0.size))
    x_res = x_res.at[0].set(x0)

    # append initial control, which is only applied to vstate_irl
    control_arr = jnp.vstack([control0.reshape(1, -1), control.control])

    def comp_uv_irl(x: jax.Array, u: jax.Array) -> jax.Array:
        assert x.shape == (42,)
        assert u.shape == (6,)

        xr = x[:12]

        euler = xr[3: 6]
        euler_dot = xr[9: 12]
        euler_all = jnp.concatenate([euler, euler_dot])
        R = comp.rot(*euler)

        u_acc = jnp.ravel(R.T @ (u[:3] + const.gravity))
        u_omega = jnp.ravel(comp.angle_vel(*euler_all))
        return jnp.concatenate([u_acc, u_omega])

    # for i in range(1, x_res.shape[0]):
    def for_body(i: int, x_res: jax.Array) -> jax.Array:
        ur = control_arr[i]
        uv_irl = comp_uv_irl(x_res[i - 1], control_arr[i - 1])
        uv_sim = jnp.concatenate([acc_ref[i - 1], omega_ref[i - 1]])
        u = jnp.concatenate([ur, uv_irl, uv_sim])
        assert u.shape == (18,)

        x_res_i = E0 @ x_res[i - 1] + E1 @ u
        return x_res.at[i].set(x_res_i)

    x_res = jax.lax.fori_loop(1, x_res.shape[0], for_body, x_res)

    xv_sim = x_res[:, 12 + 15:]
    xv_irl = jnp.empty((n + 1, 15))
    xv_irl = xv_irl.at[:-1, :].set(x_res[1:, 12: 12 + 15])
    uv_irl = comp_uv_irl(x_res[-1], control_arr[-1])
    xv_irl_1 = E0_ves @ xv_irl[-2] + E1_ves @ uv_irl
    xv_irl = xv_irl.at[-1].set(xv_irl_1)

    yv_irl = jnp.transpose(C_ves @ xv_irl.T)
    yv_sim = jnp.transpose(C_ves @ xv_sim.T)

    # xr = x_res[:, :12]
    # rstate = utils.RState(xr)
    # vstate_irl = utils.VState(xv_irl, yv_irl)
    # vstate_sim = utils.VState(xv_sim, yv_sim)
    # return rstate, vstate_irl, vstate_sim

    diff = jnp.ravel(yv_irl[1:] - yv_sim[1:])
    return diff @ diff

@jax.jit
def monolithic_tracking_error_flat(
    acc_ref: jax.Array,
    omega_ref: jax.Array,
    rstate0: jax.Array,
    vstate0_irl: jax.Array,
    vstate0_sim: jax.Array,
    control0: jax.Array,
    control_flat: jax.Array,
) -> jax.Array:
    return monolithic_tracking_error_jax(
        acc_ref=acc_ref,
        omega_ref=omega_ref,
        rstate0=rstate0,
        vstate0_irl=vstate0_irl,
        vstate0_sim=vstate0_sim,
        control0=control0,
        control=utils.Control.from_flat(control_flat),
    )


monolithic_tracking_error_and_grad = jax.jit(jax.value_and_grad(
    monolithic_tracking_error_flat, argnums=6
))

def call_jax() -> jax.Array:
    res = monolithic_tracking_error_jax(
        acc_ref=acc_ref,
        omega_ref=omega_ref,
        rstate0=rstate0,
        vstate0_irl=vstate0_irl,
        vstate0_sim=vstate0_sim,
        control0=control0,
        control=control,
    )
    res.block_until_ready()
    return res

def call_jax_grad() -> jax.Array:
    res = monolithic_tracking_error_and_grad(
        acc_ref,
        omega_ref,
        rstate0,
        vstate0_irl,
        vstate0_sim,
        control0,
        control.flatten(),
    )
    res[0].block_until_ready()
    res[1].block_until_ready()
    return res

In [ ]:
check0 = call_ref()
check1 = call_jax()
jnp.abs(check0 - check1)

In [ ]:
# get_monolithic_state_call()
# with jax.profiler.trace("/tmp/jax-trace", create_perfetto_link=True):
#     get_monolithic_state_call()

In [ ]:
call_jax()
%timeit -n 100 -r 7 call_jax()

In [ ]:
call_jax_grad()
%timeit -n 100 -r 7 call_jax_grad()

## eigen iterations (numpy)

We introduce fancy linear-algebraic update scheme.
Consider the recursive problem
$$
  x_k = E_0 x_{k - 1} + E_1 u_{k - 1},
$$
with given $u_k$ and $x_0$.
We want to update this efficiently, both in the forward pass and in the backpropagation of gradients.
The following scheme is posited.
Suppose that $E_0$ is diagonalizable, say
$$
  E_0 = P D P^{-1},
$$
with $D$ diagonal and with $P$ the corresponding eigenvectors.
If we introduce the notation $\tilde{x}_k = P^{-1} x_k$ and $\tilde{u}_k = P^{-1} E_1 u_k$, then we have the update rules
$$
  \tilde{x}_k = D \tilde{x}_{k - 1} + \tilde{u}_{k - 1}, \quad \tilde{x}_0 = P^{-1} x_0.
$$
again, with $D$ diagonal.
So, these update rules can be applied componentwise.
Simply counting floating point operations shows that this is more efficient (by a constant factor).
More importantly, the back propagation of gradients rule is very simple, because we are simply acting componentwise in our updates, most of the time.
To get our desired observed variables, we have the conversion
$$
  y_k = C P \tilde{x}_k.
$$
Numerical experiments (done in a temporary notebook) show that this update scheme is very stable (for typical horizon lengths).
So, the main question is if this is actually more efficient in a JAX compiled function, with backpropagated gradients.

In [ ]:
tmp = sci_lin.eig(const.E0_acc)
D_acc, P_acc = tmp[0], tmp[1]
P_acc_inv = np.linalg.inv(P_acc)
D_acc = D_acc.real  # type: ignore
assert np.max(np.abs(P_acc @ np.diag(D_acc.real) @ P_acc_inv - const.E0_acc)) < 1e-15

tmp = sci_lin.eig(const.E0_omega)
D_omega, P_omega = tmp[0], tmp[1]
P_omega_inv = np.linalg.inv(P_omega)
D_omega = D_omega.real  # type: ignore
assert np.max(np.abs(P_omega @ np.diag(D_omega.real) @ P_omega_inv - const.E0_omega)) < 1e-15

EP1_acc = np.linalg.inv(P_acc) @ const.E1_acc
CP_acc = const.C_acc @ P_acc
EP1_omega = np.linalg.inv(P_omega) @ const.E1_omega
CP_omega = const.C_omega @ P_omega

In [ ]:
# uninteresting computations are done in jax, because it is convenient...

def init_update(E0, E1, v0, u):
    v0 = v0.reshape(-1, E0.shape[0])
    v0 = E0 @ v0.T + E1 @ u.reshape(1, -1)
    return v0.T.flatten()

def eigen_int(D, EP1, CP, P_inv, v0, u):
    # transform into the eigen vector coordinates
    x0 = np.transpose(P_inv @ v0.reshape(-1, D.size).T)

    # control wizardry with shapes...
    # ut is a 3-tensor, with indices
    #  (references index, state component index, time index)
    ut = np.transpose(EP1 @ u.T.reshape(1, -1))  # (time, ref/state)
    ut = ut.reshape(x0.shape[0], -1, x0.shape[1])  # (ref, time, state)
    ut = np.transpose(ut, axes=(0, 2, 1))  # (ref, state, time)

    # desired final state shape: (ref, time, state)
    # iterations:
    #  initial conditions, components, time
    x = np.empty((ut.shape[0], ut.shape[1], ut.shape[2] + 1))  # (ref, state, time)
    for i in range(ut.shape[0]):
        for j in range(ut.shape[1]):
            x[i, j, 0] = x0[i, j]
            for k in range(ut.shape[2]):
                x[i, j, k + 1] = D[j] * x[i, j, k] + ut[i, j, k]
    x = np.transpose(x, axes=(0, 2, 1))  # (ref, time, state)
    x = x[:, 1:, :]  # skip initial condition

    # desired final output shape (SISO): (ref, time)
    y = np.empty((x.shape[0], x.shape[1]))
    for i in range(x.shape[0]):
        y[i] = np.squeeze(CP @ x[i].T)
    return y

def eigen_tracking_error_np(
    acc_ref: np.ndarray,
    omega_ref: np.ndarray,
    rstate0: np.ndarray,
    vstate0_irl: np.ndarray,
    vstate0_sim: np.ndarray,
    control0: np.ndarray,
    control: np.ndarray,
) -> float:
    # get irl controls
    control_jax0 = utils.Control.from_flat(jnp.vstack([control0, control]))
    control_jax1 = utils.Control.from_flat(jnp.array(control))
    rstate = utils.get_rstate(control_jax1, jnp.array(rstate0))
    acc_irl = jax.vmap(
        lambda r, u: utils.rot(r).T @ (jnp.array([u.x, u.y, u.z]) + const.gravity)
    )(rstate, control_jax0)
    omega_irl = jax.vmap(utils.angle_vel)(rstate)

    # partition
    a_num = 3 * const.E0_acc.shape[0]
    v0_irl_a = vstate0_irl[:a_num]
    v0_irl_w = vstate0_irl[a_num:]
    v0_sim_a = vstate0_sim[:a_num]
    v0_sim_w = vstate0_sim[a_num:]

    # initial irl update (for closed feedback)
    v0_irl_a = init_update(const.E0_acc, const.E1_acc, v0_irl_a, acc_irl[0])
    v0_irl_w = init_update(const.E0_omega, const.E1_omega, v0_irl_w, omega_irl[0])
    acc_irl = acc_irl[1:]
    omega_irl = omega_irl[1:]

    # setup general states and controls
    v0_a = np.concatenate([v0_irl_a, v0_sim_a])
    v0_w = np.concatenate([v0_irl_w, v0_sim_w])
    u_a = np.hstack([acc_irl, acc_ref])
    u_w = np.hstack([omega_irl, omega_ref])

    # integrate
    y_a = eigen_int(D_acc, EP1_acc, CP_acc, P_acc_inv, v0_a, u_a)
    y_w = eigen_int(D_omega, EP1_omega, CP_omega, P_omega_inv, v0_w, u_w)

    # res
    y_irl = np.vstack([y_a[:3], y_w[:3]])
    y_sim = np.vstack([y_a[3:], y_w[3:]])
    diff = np.ravel(y_irl - y_sim)
    return float(diff @ diff)

def call_eigen_np() -> float:
    return eigen_tracking_error_np(
        acc_ref=np.array(acc_ref),
        omega_ref=np.array(omega_ref),
        rstate0=np.array(rstate0),
        vstate0_irl=np.array(vstate0_irl),
        vstate0_sim=np.array(vstate0_sim),
        control0=np.array(control0),
        control=np.array(control.control),
    )


In [ ]:
check0 = call_ref()
check1 = call_eigen_np()
check0 - check1

## eigen iterations (jax)

In [ ]:
def eigen_int_jax(
    D: jax.Array | np.ndarray,
    EP1: jax.Array | np.ndarray,
    CP: jax.Array | np.ndarray,
    P_inv: jax.Array | np.ndarray,
    v0: jax.Array,
    u: jax.Array,
) -> jax.Array:
    assert len(D.shape) == 1
    assert len(EP1.shape) == 2
    assert EP1.shape[1] == 1
    assert D.size == EP1.shape[0]
    assert P_inv.shape[0] == P_inv.shape[1] and P_inv.shape[0] == EP1.shape[0]
    assert len(v0.shape) == 1
    assert v0.size % P_inv.shape[0] == 0
    assert len(u.shape) == 2
    assert u.shape[1] == v0.size // P_inv.shape[0]
    assert u.size > 0

    # transform into the eigen vector coordinates
    x0 = jnp.transpose(P_inv @ v0.reshape(-1, D.size).T)

    # control wizardry with shapes...
    # ut is a 3-tensor, with indices
    #  (references index, state component index, time index)
    ut = jnp.transpose(EP1 @ u.T.reshape(1, -1))  # (time, ref/state)
    ut = ut.reshape(x0.shape[0], -1, x0.shape[1])  # (ref, time, state)
    ut = jnp.transpose(ut, axes=(0, 2, 1))  # (ref, state, time)

    # desired final state shape: (ref, time, state)
    # iterations:
    #  initial conditions, components, time

    def eigen_update(d, x0, u):
        x1 = d * x0 + u
        return x1, x1

    def scan_eigen_update(x0, d, u):
        part_eigen_update = functools.partial(eigen_update, d)
        return jax.lax.scan(part_eigen_update, x0, u)[1]

    d = jnp.tile(D, reps=(ut.shape[0], 1))
    x = jax.vmap(jax.vmap(scan_eigen_update))(x0, d, ut)
    assert isinstance(x, jax.Array)
    x = jnp.transpose(x, axes=(0, 2, 1))  # (ref, time, state)

    # desired final output shape (SISO): (ref, time)
    y = jax.vmap(lambda x: jnp.squeeze(CP @ x.T))(x)
    return y

@jax.jit
def eigen_tracking_error_jax(
    acc_ref: jax.Array,
    omega_ref: jax.Array,
    rstate0: jax.Array,
    vstate0_irl: jax.Array,
    vstate0_sim: jax.Array,
    control0: jax.Array,
    control: utils.Control,
) -> jax.Array:
    # get irl controls
    control_with0 = utils.Control.from_flat(jnp.vstack([control0, control.control]))
    rstate = utils.get_rstate(control, jnp.array(rstate0))
    acc_irl = jax.vmap(
        lambda r, u: utils.rot(r).T @ (jnp.array([u.x, u.y, u.z]) + const.gravity)
    )(rstate, control_with0)
    omega_irl = jax.vmap(utils.angle_vel)(rstate)

    # partition
    a_num = 3 * const.E0_acc.shape[0]
    v0_irl_a = vstate0_irl[:a_num]
    v0_irl_w = vstate0_irl[a_num:]
    v0_sim_a = vstate0_sim[:a_num]
    v0_sim_w = vstate0_sim[a_num:]

    # initial irl update (for closed feedback)
    v0_irl_a = init_update(const.E0_acc, const.E1_acc, v0_irl_a, acc_irl[0])
    v0_irl_w = init_update(const.E0_omega, const.E1_omega, v0_irl_w, omega_irl[0])
    acc_irl = acc_irl[1:]
    omega_irl = omega_irl[1:]

    # setup general states and controls
    v0_a = jnp.concatenate([v0_irl_a, v0_sim_a])
    v0_w = jnp.concatenate([v0_irl_w, v0_sim_w])
    u_a = jnp.hstack([acc_irl, acc_ref])
    u_w = jnp.hstack([omega_irl, omega_ref])

    # integrate
    y_a = eigen_int_jax(D_acc, EP1_acc, CP_acc, P_acc_inv, v0_a, u_a)
    y_w = eigen_int_jax(D_omega, EP1_omega, CP_omega, P_omega_inv, v0_w, u_w)

    # res
    y_irl = jnp.vstack([y_a[:3], y_w[:3]])
    y_sim = jnp.vstack([y_a[3:], y_w[3:]])
    diff = jnp.ravel(y_irl - y_sim)
    return jnp.squeeze(diff @ diff)

@jax.jit
def eigen_tracking_error_flat(
    acc_ref: jax.Array,
    omega_ref: jax.Array,
    rstate0: jax.Array,
    vstate0_irl: jax.Array,
    vstate0_sim: jax.Array,
    control0: jax.Array,
    control_flat: jax.Array,
) -> jax.Array:
    control = utils.Control.from_flat(control_flat)
    return eigen_tracking_error_jax(
        acc_ref=acc_ref,
        omega_ref=omega_ref,
        rstate0=rstate0,
        vstate0_irl=vstate0_irl,
        vstate0_sim=vstate0_sim,
        control0=control0,
        control=control,
    )

eigen_tracking_error_and_grad = jax.jit(jax.value_and_grad(
    eigen_tracking_error_flat, argnums=6
))

def call_eigen_jax() -> jax.Array:
    res = eigen_tracking_error_jax(
        acc_ref=acc_ref,
        omega_ref=omega_ref,
        rstate0=rstate0,
        vstate0_irl=vstate0_irl,
        vstate0_sim=vstate0_sim,
        control0=control0,
        control=control,
    )
    res.block_until_ready()
    return res

def call_eigen_jax_grad() -> jax.Array:
    res = eigen_tracking_error_and_grad(
        acc_ref,
        omega_ref,
        rstate0,
        vstate0_irl,
        vstate0_sim,
        control0,
        control.flatten(),
    )
    res[0].block_until_ready()
    res[1].block_until_ready()
    return res

In [ ]:
check0 = call_ref()
check1 = call_eigen_jax()
check0 - check1

In [ ]:
call_eigen_jax()
%timeit -n 1000 -r 7 call_eigen_jax()

In [ ]:
call_eigen_jax()
%timeit -n 1000 -r 7 call_eigen_jax_grad()